This is a **CLEAN** but vivid version of codes (along with visualizations) 
for analyzing the experiment ***freight collaboration-chessboard*** results

In [11]:
from dataclasses import dataclass
from enum import Enum
import os
import sys
import pandas as pd
from pathlib import Path
from itertools import product
# Use repo-relative path so it works on other machines
notebook_dir = Path.cwd() / "python" / "test"
if notebook_dir.exists() and str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
import matsim
import matsim_output_reader
import metric_anls
# --- reload the module to reflect any changes made during development ---
import importlib
importlib.reload(matsim_output_reader)
importlib.reload(metric_anls)

<module 'metric_anls' from '/Volumes/External/gitProj/matsim-libs-2024/python/test/metric_anls.py'>

# Configuration

In [3]:
# Resolve analysis path relative to repo root
repo_root = notebook_dir.parents[1]  # .../matsim-libs-2024
anls_path = repo_root / "output" / "chessboardCarrierReceiverCollab"

class DepotLocation(Enum):
    INSIDE = 'center'
    OUTSIDE = 'left'

class ReceiverDistribution(Enum):
    DISPERSED = 'DISPERSED'
    CLUSTERED = 'CLUSTERED'
    RANDOM = 'FULLY_RANDOM'

ALLOCATION_FACTOR = 0.8
#--- Penalty ---#
PENALTY_LIST = [0, 0.0003, 0.0008, 0.0014, 0.0028, 0.0056,
                0.0098, 0.014, 0.0167, 0.0222, 0.028]
PENALTY_LIST_SCALE = [round(x * 3600) for x in PENALTY_LIST]  # scale to avoid float precision issues
PENALTY_LIST_SCALE[-1] = 100 
PENALTY_DICT = {k: v for k, v in zip(PENALTY_LIST, PENALTY_LIST_SCALE)}
#--- Penalty ---#

TOTAL_INSTANCES = 50


In [6]:
#--- Generate keywords for each scenario ---#
# Example tuple: ("center", "dispersed", 0)
scenario_keywords = [
    (depot, receiver, penalty)
    for depot, receiver, penalty in product([DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value],
                                            [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value], 
                                            PENALTY_LIST)
]
scenario_keywords[:3]

[('center', 'DISPERSED', 0),
 ('center', 'DISPERSED', 0.0003),
 ('center', 'DISPERSED', 0.0008)]

# Test

In [12]:
test_folder = os.listdir(anls_path)[5]
print(f"Analyzing results from folder: {test_folder}")
test_carriers_df, test_shipments_df = matsim_output_reader.read_carriers(
    os.path.join(anls_path, test_folder, 'output_carriers.xml.gz')
)
test_receivers_dict = matsim_output_reader.read_receivers(
    os.path.join(anls_path, test_folder, 'receivers.xml.gz'), (6,8))

test_iter0_carrier_df, test_iter0_shipment_df = matsim_output_reader.read_iter0_carriers(
    os.path.join(anls_path, test_folder), False)

test_iter0_carrier_score_dict = matsim_output_reader.read_iter0_carriers(
    os.path.join(anls_path, test_folder))

Analyzing results from folder: center-FULLY_RANDOM-penSweep-af0.80-p0.0014-exactShapley-i46


In [22]:
matsim_network = matsim.read_network(os.path.join(anls_path, test_folder, 'output_network.xml.gz'))
network_link_df = matsim_network.links
network_node_df = matsim_network.nodes

In [51]:
network_link_df

,length,freespeed,capacity,permlanes,oneway,modes,link_id,from_node,to_node
0,1000.0,7.5,10.0,1.0,1,car,"i(1,0)","(0,0)","(1,0)"
1,1000.0,7.5,10.0,1.0,1,car,"i(1,1)R","(1,1)","(0,1)"
2,1000.0,7.5,10.0,1.0,1,car,"i(1,2)","(0,2)","(1,2)"
3,1000.0,7.5,10.0,1.0,1,car,"i(1,3)R","(1,3)","(0,3)"
4,1000.0,7.5,10.0,1.0,1,car,"i(1,4)","(0,4)","(1,4)"
...,...,...,...,...,...,...,...,...,...
175,1000.0,7.5,10.0,1.0,1,car,"j(9,5)","(9,4)","(9,5)"
176,1000.0,7.5,10.0,1.0,1,car,"j(9,6)","(9,5)","(9,6)"
177,1000.0,7.5,10.0,1.0,1,car,"j(9,7)","(9,6)","(9,7)"
178,1000.0,7.5,10.0,1.0,1,car,"j(9,8)","(9,7)","(9,8)"


## Test for reader

In [13]:
test_carriers_df

,carrier_id,depot_link_ids,depot_link_id,vehicle_ids,num_vehicles,carrier_score,num_shipments,iter0_carrier_score,iter0_num_vehicles
0,carrier1,"[i(5,5)R, i(5,5)R]","i(5,5)R","[lightVan1, heavyVan1]",2,586.295312,10,575.763,2


In [7]:
test_shipments_df

,carrier_id,shipment_id,pickup_link_id,delivery_link_id,size,start_pickup,end_pickup,start_delivery,end_delivery,pickup_service_time,delivery_service_time
0,carrier1,Orderreceiver_001,"i(5,5)R","i(7,2)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
1,carrier1,Orderreceiver_012,"i(5,5)R","i(4,6)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
2,carrier1,Orderreceiver_023,"i(5,5)R","i(6,4)",500,00:00:00,596523:14:07,06:00:00,09:00:00,00:00:00,00:20:00
3,carrier1,Orderreceiver_034,"i(5,5)R","i(5,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
4,carrier1,Orderreceiver_045,"i(5,5)R","j(7,3)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
5,carrier1,Orderreceiver_056,"i(5,5)R","j(7,5)",500,00:00:00,596523:14:07,06:00:00,11:00:00,00:00:00,00:20:00
6,carrier1,Orderreceiver_067,"i(5,5)R","i(4,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
7,carrier1,Orderreceiver_078,"i(5,5)R","i(5,3)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
8,carrier1,Orderreceiver_089,"i(5,5)R","i(6,3)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
9,carrier1,Orderreceiver_0910,"i(5,5)R","i(3,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00


In [8]:
test_iter0_carrier_df

,carrier_id,depot_link_ids,depot_link_id,vehicle_ids,num_vehicles,carrier_score,num_shipments
0,carrier1,"[i(5,5)R, i(5,5)R]","i(5,5)R","[lightVan1, heavyVan1]",2,575.763,10


In [9]:
test_iter0_shipment_df

,carrier_id,shipment_id,pickup_link_id,delivery_link_id,size,start_pickup,end_pickup,start_delivery,end_delivery,pickup_service_time,delivery_service_time
0,carrier1,Orderreceiver_001,"i(5,5)R","i(7,2)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
1,carrier1,Orderreceiver_012,"i(5,5)R","i(4,6)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
2,carrier1,Orderreceiver_023,"i(5,5)R","i(6,4)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
3,carrier1,Orderreceiver_034,"i(5,5)R","i(5,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
4,carrier1,Orderreceiver_045,"i(5,5)R","j(7,3)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
5,carrier1,Orderreceiver_056,"i(5,5)R","j(7,5)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
6,carrier1,Orderreceiver_067,"i(5,5)R","i(4,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
7,carrier1,Orderreceiver_078,"i(5,5)R","i(5,3)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
8,carrier1,Orderreceiver_089,"i(5,5)R","i(6,3)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
9,carrier1,Orderreceiver_0910,"i(5,5)R","i(3,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00


In [50]:
test_iter0_carrier_score_dict

{'carrier1': 575.7629999999999}

In [10]:
test_receivers_dict

{'collaborative_receivers': [{'id': 'receiver_02',
   'time_window': (6, 9),
   'time_window_str': '06:00:00 - 09:00:00',
   'score': -89.984522800001},
  {'id': 'receiver_05',
   'time_window': (6, 11),
   'time_window_str': '06:00:00 - 11:00:00',
   'score': -100.28670040000013}],
 'non_collaborative_receivers': [{'id': 'receiver_00',
   'time_window': (6, 8),
   'time_window_str': '06:00:00 - 08:00:00',
   'score': -100.0},
  {'id': 'receiver_01',
   'time_window': (6, 8),
   'time_window_str': '06:00:00 - 08:00:00',
   'score': -100.0},
  {'id': 'receiver_03',
   'time_window': (6, 8),
   'time_window_str': '06:00:00 - 08:00:00',
   'score': -100.0},
  {'id': 'receiver_04',
   'time_window': (6, 8),
   'time_window_str': '06:00:00 - 08:00:00',
   'score': -100.0},
  {'id': 'receiver_06',
   'time_window': (6, 8),
   'time_window_str': '06:00:00 - 08:00:00',
   'score': -100.0},
  {'id': 'receiver_07',
   'time_window': (6, 8),
   'time_window_str': '06:00:00 - 08:00:00',
   'score'

In [15]:
test_collaboration_data = matsim_output_reader.read_collaboration_allocation_data(os.path.join(anls_path, test_folder,))
test_collaboration_data

{'receiver_05': 18.92062319999932,
 'carrier1': 9.460311599999656,
 'receiver_02': 18.92062319999932}

## Test for metric analysis

In [17]:
test_fv_travel_chains = metric_anls.derive_freight_vehicle_travel_chains(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz')
)
test_fv_travel_chains

{'freight_carrier1_veh_lightVan1_2': ['i(5,5)R',
  'i(4,5)R',
  'j(3,6)',
  'i(4,6)',
  'i(5,6)',
  'j(5,7)',
  'i(5,7)R',
  'i(4,7)R',
  'i(3,7)R',
  'j(2,7)R',
  'i(3,6)',
  'i(4,6)',
  'j(4,6)R',
  'j(4,5)R',
  'i(5,4)',
  'j(5,5)',
  'i(5,5)R'],
 'freight_carrier1_veh_lightVan1_1': ['i(5,5)R',
  'j(4,5)R',
  'i(5,4)',
  'i(6,4)',
  'j(6,4)R',
  'j(6,3)R',
  'i(7,2)',
  'j(7,3)',
  'i(7,3)R',
  'i(6,3)R',
  'i(5,3)R',
  'i(4,3)R',
  'j(3,4)',
  'i(4,4)',
  'i(5,4)',
  'i(6,4)',
  'i(7,4)',
  'j(7,5)',
  'i(7,5)R',
  'i(6,5)R',
  'i(5,5)R']}

In [18]:
test_iter0_fv_travel_chains = metric_anls.derive_freight_vehicle_travel_chains(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz')
)
test_iter0_fv_travel_chains

{'freight_carrier1_veh_lightVan1_2': ['i(5,5)R',
  'i(4,5)R',
  'j(3,6)',
  'i(4,6)',
  'i(5,6)',
  'j(5,7)',
  'i(5,7)R',
  'i(4,7)R',
  'i(3,7)R',
  'j(2,7)R',
  'i(3,6)',
  'i(4,6)',
  'j(4,6)R',
  'j(4,5)R',
  'i(5,4)',
  'i(6,4)',
  'i(7,4)',
  'j(7,5)',
  'i(7,5)R',
  'i(6,5)R',
  'i(5,5)R'],
 'freight_carrier1_veh_lightVan1_1': ['i(5,5)R',
  'j(4,5)R',
  'i(5,4)',
  'i(6,4)',
  'j(6,4)R',
  'i(6,3)R',
  'i(5,3)R',
  'j(4,3)R',
  'i(5,2)',
  'i(6,2)',
  'i(7,2)',
  'j(7,3)',
  'j(7,4)',
  'j(7,5)',
  'i(7,5)R',
  'i(6,5)R',
  'i(5,5)R']}

In [26]:
test_freight_vkt = metric_anls.compute_freight_vehicle_km_traveled(
    test_fv_travel_chains,
    network_link_df,
)
test_freight_vkt

{'freight_carrier1_veh_lightVan1_2': np.float64(17000.0),
 'freight_carrier1_veh_lightVan1_1': np.float64(21000.0)}

In [27]:
test_iter0_fv_vkt = metric_anls.compute_freight_vehicle_km_traveled(
    test_iter0_fv_travel_chains,
    network_link_df,
)
test_iter0_fv_vkt

{'freight_carrier1_veh_lightVan1_2': np.float64(21000.0),
 'freight_carrier1_veh_lightVan1_1': np.float64(17000.0)}

In [28]:
test_fv_travel_times = metric_anls.compute_freight_vehicle_travel_times(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz')
)
test_fv_travel_times

{'freight_carrier1_veh_lightVan1_2': np.float64(10103.0),
 'freight_carrier1_veh_lightVan1_1': np.float64(12681.0)}

In [29]:
test_iter0_fv_travel_times = metric_anls.compute_freight_vehicle_travel_times(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz')
)
test_iter0_fv_travel_times

{'freight_carrier1_veh_lightVan1_2': np.float64(11480.0),
 'freight_carrier1_veh_lightVan1_1': np.float64(11078.0)}

In [35]:
test_shipment_travel_timeAndDist = metric_anls.compute_freight_shipment_travel_distance_and_time(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz'),
    network_link_df
)
test_shipment_travel_timeAndDist

,shipment_id,carrier_id,vehicle_id,pickup_link,delivery_link,pickup_time,delivery_time,travel_time_seconds,travel_distance_km,capacity_demand
0,Orderreceiver_045,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","j(7,3)",18000.0,22934.0,4934.0,15.0,500.0
1,Orderreceiver_012,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(4,6)",18000.0,25872.0,7872.0,19.0,500.0
2,Orderreceiver_034,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(5,7)R",18001.0,18808.0,807.0,10.0,500.0
3,Orderreceiver_001,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","i(7,2)",18001.0,19170.0,1169.0,12.0,500.0
4,Orderreceiver_0910,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(3,7)R",18002.0,24269.0,6267.0,13.0,500.0
5,Orderreceiver_078,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","i(5,3)R",18002.0,25738.0,7736.0,18.0,500.0
6,Orderreceiver_067,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(4,7)R",18003.0,22934.0,4931.0,10.0,500.0
7,Orderreceiver_089,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","i(6,3)R",18003.0,24403.0,6400.0,15.0,500.0
8,Orderreceiver_023,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","i(6,4)",18364.0,27609.0,9245.0,22.0,500.0
9,Orderreceiver_056,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","j(7,5)",18365.0,29078.0,10713.0,24.0,500.0


In [39]:
print("Total travel time (seconds):", test_shipment_travel_timeAndDist['travel_time_seconds'].sum())
print("Total shipment travel distance (kilometers):", test_shipment_travel_timeAndDist['travel_distance_km'].sum())

Total travel time (seconds): 60074.0
Total shipment travel distance (kilometers): 158.0


In [36]:
test_iter0_shipment_travel_timeAndDist = metric_anls.compute_freight_shipment_travel_distance_and_time(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz'),
    network_link_df
)
test_iter0_shipment_travel_timeAndDist

,shipment_id,carrier_id,vehicle_id,pickup_link,delivery_link,pickup_time,delivery_time,travel_time_seconds,travel_distance_km,capacity_demand
0,Orderreceiver_089,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","i(6,3)R",18000.0,18675.0,675.0,11.0,500.0
1,Orderreceiver_023,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(6,4)",18000.0,27609.0,9609.0,25.0,500.0
2,Orderreceiver_034,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(5,7)R",18001.0,19169.0,1168.0,11.0,500.0
3,Orderreceiver_001,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","i(7,2)",18001.0,24671.0,6670.0,17.0,500.0
4,Orderreceiver_067,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(4,7)R",18002.0,22934.0,4932.0,12.0,500.0
5,Orderreceiver_056,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","j(7,5)",18002.0,27475.0,9473.0,21.0,500.0
6,Orderreceiver_045,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","j(7,3)",18003.0,26006.0,8003.0,17.0,500.0
7,Orderreceiver_0910,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(3,7)R",18003.0,24269.0,6266.0,13.0,500.0
8,Orderreceiver_012,carrier1,freight_carrier1_veh_lightVan1_2,"i(5,5)R","i(4,6)",18004.0,25872.0,7868.0,16.0,500.0
9,Orderreceiver_078,carrier1,freight_carrier1_veh_lightVan1_1,"i(5,5)R","i(5,3)R",18004.0,22934.0,4930.0,9.0,500.0


In [40]:
print("Total travel time (seconds):", test_iter0_shipment_travel_timeAndDist['travel_time_seconds'].sum())
print("Total shipment travel distance (kilometers):", test_iter0_shipment_travel_timeAndDist['travel_distance_km'].sum())   


Total travel time (seconds): 59594.0
Total shipment travel distance (kilometers): 152.0


In [46]:
test_missed_tw = metric_anls.analyse_missed_time_windows(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz'),
    test_shipments_df,
)
test_missed_tw

,shipment_id,time_window_start,time_window_end,actual_delivery_time,is_on_time,is_early,is_late,deviation_seconds,deviation_minutes,deviation_with_sign (min)
0,Orderreceiver_001,21600.0,28800.0,19170.0,False,True,False,2430.0,40.500000,-40.500000
1,Orderreceiver_012,21600.0,28800.0,25872.0,True,False,False,0.0,0.000000,0.000000
2,Orderreceiver_023,21600.0,32400.0,27609.0,True,False,False,0.0,0.000000,0.000000
3,Orderreceiver_034,21600.0,28800.0,18808.0,False,True,False,2792.0,46.533333,-46.533333
4,Orderreceiver_045,21600.0,28800.0,22934.0,True,False,False,0.0,0.000000,0.000000
5,Orderreceiver_056,21600.0,39600.0,29078.0,True,False,False,0.0,0.000000,0.000000
6,Orderreceiver_067,21600.0,28800.0,22934.0,True,False,False,0.0,0.000000,0.000000
7,Orderreceiver_078,21600.0,28800.0,25738.0,True,False,False,0.0,0.000000,0.000000
8,Orderreceiver_089,21600.0,28800.0,24403.0,True,False,False,0.0,0.000000,0.000000
9,Orderreceiver_0910,21600.0,28800.0,24269.0,True,False,False,0.0,0.000000,0.000000


In [47]:
test_iter0_missed_tw = metric_anls.analyse_missed_time_windows(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz'),
    test_iter0_shipment_df,
)
test_iter0_missed_tw

,shipment_id,time_window_start,time_window_end,actual_delivery_time,is_on_time,is_early,is_late,deviation_seconds,deviation_minutes,deviation_with_sign (min)
0,Orderreceiver_001,21600.0,28800.0,24671.0,True,False,False,0.0,0.000000,0.000000
1,Orderreceiver_012,21600.0,28800.0,25872.0,True,False,False,0.0,0.000000,0.000000
2,Orderreceiver_023,21600.0,28800.0,27609.0,True,False,False,0.0,0.000000,0.000000
3,Orderreceiver_034,21600.0,28800.0,19169.0,False,True,False,2431.0,40.516667,-40.516667
4,Orderreceiver_045,21600.0,28800.0,26006.0,True,False,False,0.0,0.000000,0.000000
5,Orderreceiver_056,21600.0,28800.0,27475.0,True,False,False,0.0,0.000000,0.000000
6,Orderreceiver_067,21600.0,28800.0,22934.0,True,False,False,0.0,0.000000,0.000000
7,Orderreceiver_078,21600.0,28800.0,22934.0,True,False,False,0.0,0.000000,0.000000
8,Orderreceiver_089,21600.0,28800.0,18675.0,False,True,False,2925.0,48.750000,-48.750000
9,Orderreceiver_0910,21600.0,28800.0,24269.0,True,False,False,0.0,0.000000,0.000000


# Aggregate Analysis

Use `agg_anls` module to process all scenarios and generate outputs.

In [4]:
# Import the aggregate analysis module
import agg_anls
importlib.reload(agg_anls)

# Configuration for batch processing
INPUT_PATH = str(anls_path)
OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "clean")

# Define which scenarios to process
DEPOT_LOCATIONS = [DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value]
RECEIVER_DISTRIBUTIONS = [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value]
ORIGINAL_TW = (6, 8)  # Original time window in hours
LAST_ITER = 50  # Last iteration number

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Depot locations: {DEPOT_LOCATIONS}")
print(f"Receiver distributions: {RECEIVER_DISTRIBUTIONS}")
print(f"Penalties: {PENALTY_LIST}")

Input path: /Volumes/External/gitProj/matsim-libs-2024/output/chessboardCarrierReceiverCollab
Output path: /Volumes/External/gitProj/matsim-libs-2024/data/freightChessboardRC/clean
Depot locations: ['center', 'left']
Receiver distributions: ['DISPERSED', 'CLUSTERED']
Penalties: [0, 0.0003, 0.0008, 0.0014, 0.0028, 0.0056, 0.0098, 0.014, 0.0167, 0.0222, 0.028]


## Test Single Scenario Processing

Before running batch processing, test with a single scenario to verify the functions work correctly.

In [53]:
# Test parsing folder name
test_folder_name = test_folder
test_config = agg_anls.parse_folder_name(test_folder_name, str(anls_path))
print(f"Parsed config: {test_config}")

Parsed config: ScenarioConfig(depot_location='center', receiver_distribution='FULLY_RANDOM', allocation_factor=0.8, penalty=0.0014, instance=46, folder_name='center-FULLY_RANDOM-penSweep-af0.80-p0.0014-exactShapley-i46', folder_path='/Volumes/External/gitProj/matsim-libs-2024/output/chessboardCarrierReceiverCollab/center-FULLY_RANDOM-penSweep-af0.80-p0.0014-exactShapley-i46')


In [54]:
# Build network graph once for efficiency
network_graph = agg_anls.build_network_graph(network_link_df, network_node_df)
print(f"Network graph: {network_graph.number_of_nodes()} nodes, {network_graph.number_of_edges()} edges")

Network graph: 102 nodes, 180 edges


In [57]:
# Test compute_scenario_metrics for single scenario
if test_config:
    test_metrics = agg_anls.compute_scenario_metrics(
        test_config,
        network_link_df,
        network_node_df,
        network_graph,
        original_tw=ORIGINAL_TW,
        last_iter=LAST_ITER,
        compute_network_distances=True
    )
# Display as DataFrame for better visualization
pd.DataFrame([test_metrics])

,depot_location,receiver_distribution,allocation_factor,penalty,instance,fleet_size,final_carrier_score,iter0_carrier_score,num_collaborative_receivers,num_non_collab_receivers,...,VKT_km,iter0_VKT_km,VTT_seconds,iter0_VTT_seconds,TKT_tonkm,iter0_TKT_tonkm,mean_receiver_dist_to_depot_euclidean_km,mean_receiver_dist_to_depot_network_km,clustering_index_euclidean_km,clustering_index_network_km
0,center,FULLY_RANDOM,0.8,0.0014,46,2,586.295312,575.763,2,8,...,38.0,38.0,22784.0,22558.0,79000.0,76000.0,2.538482,4.4,1.082843,1.9


In [58]:
# Test compute_merged_shipment_df
if test_config:
    test_merged_shipment_df = agg_anls.compute_merged_shipment_df(test_config, network_link_df)
    print(f"Merged shipment DataFrame shape: {test_merged_shipment_df.shape}")
test_merged_shipment_df

Merged shipment DataFrame shape: (10, 42)


,carrier_id,shipment_id,pickup_link_id,delivery_link_id,size,start_pickup,end_pickup,start_delivery,end_delivery,pickup_service_time,...,iter0_delivery_link,iter0_pickup_time,iter0_delivery_time,iter0_travel_time_seconds,iter0_travel_distance_km,iter0_capacity_demand,depot_location,receiver_distribution,penalty,instance
0,carrier1,Orderreceiver_001,"i(5,5)R","i(7,2)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"i(7,2)",18001.0,24671.0,6670.0,17.0,500.0,center,FULLY_RANDOM,0.0014,46
1,carrier1,Orderreceiver_012,"i(5,5)R","i(4,6)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"i(4,6)",18004.0,25872.0,7868.0,16.0,500.0,center,FULLY_RANDOM,0.0014,46
2,carrier1,Orderreceiver_023,"i(5,5)R","i(6,4)",500,00:00:00,596523:14:07,06:00:00,09:00:00,00:00:00,...,"i(6,4)",18000.0,27609.0,9609.0,25.0,500.0,center,FULLY_RANDOM,0.0014,46
3,carrier1,Orderreceiver_034,"i(5,5)R","i(5,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"i(5,7)R",18001.0,19169.0,1168.0,11.0,500.0,center,FULLY_RANDOM,0.0014,46
4,carrier1,Orderreceiver_045,"i(5,5)R","j(7,3)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"j(7,3)",18003.0,26006.0,8003.0,17.0,500.0,center,FULLY_RANDOM,0.0014,46
5,carrier1,Orderreceiver_056,"i(5,5)R","j(7,5)",500,00:00:00,596523:14:07,06:00:00,11:00:00,00:00:00,...,"j(7,5)",18002.0,27475.0,9473.0,21.0,500.0,center,FULLY_RANDOM,0.0014,46
6,carrier1,Orderreceiver_067,"i(5,5)R","i(4,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"i(4,7)R",18002.0,22934.0,4932.0,12.0,500.0,center,FULLY_RANDOM,0.0014,46
7,carrier1,Orderreceiver_078,"i(5,5)R","i(5,3)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"i(5,3)R",18004.0,22934.0,4930.0,9.0,500.0,center,FULLY_RANDOM,0.0014,46
8,carrier1,Orderreceiver_089,"i(5,5)R","i(6,3)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"i(6,3)R",18000.0,18675.0,675.0,11.0,500.0,center,FULLY_RANDOM,0.0014,46
9,carrier1,Orderreceiver_0910,"i(5,5)R","i(3,7)R",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,...,"i(3,7)R",18003.0,24269.0,6266.0,13.0,500.0,center,FULLY_RANDOM,0.0014,46


In [59]:
# Test create_geo_dataframe
if test_config:
    test_geo_df = agg_anls.create_geo_dataframe(
        test_config,
        network_node_df,
        network_link_df,
        original_tw=ORIGINAL_TW,
        last_iter=LAST_ITER
    )
    print(f"GeoDataFrame shape: {test_geo_df.shape}")
test_geo_df

GeoDataFrame shape: (11, 11)


,id,type,is_collaborative,payoff_allocation,relaxed_tw_hours,link_id,geometry,depot_location,receiver_distribution,penalty,instance
0,carrier1,carrier,True,9.460312,0.0,"i(5,5)R",POINT (4000 5000),center,FULLY_RANDOM,0.0014,46
1,receiver_02,receiver,True,18.920623,1.0,"i(6,4)",POINT (6000 4000),center,FULLY_RANDOM,0.0014,46
2,receiver_05,receiver,True,18.920623,3.0,"j(7,5)",POINT (7000 5000),center,FULLY_RANDOM,0.0014,46
3,receiver_00,receiver,False,0.000000,0.0,"i(7,2)",POINT (7000 2000),center,FULLY_RANDOM,0.0014,46
4,receiver_01,receiver,False,0.000000,0.0,"i(4,6)",POINT (4000 6000),center,FULLY_RANDOM,0.0014,46
5,receiver_03,receiver,False,0.000000,0.0,"i(5,7)R",POINT (4000 7000),center,FULLY_RANDOM,0.0014,46
6,receiver_04,receiver,False,0.000000,0.0,"j(7,3)",POINT (7000 3000),center,FULLY_RANDOM,0.0014,46
7,receiver_06,receiver,False,0.000000,0.0,"i(4,7)R",POINT (3000 7000),center,FULLY_RANDOM,0.0014,46
8,receiver_07,receiver,False,0.000000,0.0,"i(5,3)R",POINT (4000 3000),center,FULLY_RANDOM,0.0014,46
9,receiver_08,receiver,False,0.000000,0.0,"i(6,3)R",POINT (5000 3000),center,FULLY_RANDOM,0.0014,46


## Batch Processing

Run the batch processing for all scenarios. This will:
1. Process each matching scenario folder
2. Compute metrics and save to `metrics.csv.gz`
3. Create merged shipment data and save to `shipments.csv.gz`
4. Create GeoDataFrame and save to `geo_data.geojson`
5. Save combined metrics to `all_scenarios_metrics.csv.gz`

In [65]:
# Option 1: Use quick_analyze with default parameters
# all_metrics_df = agg_anls.quick_analyze(
#     input_path=INPUT_PATH,
#     output_path=OUTPUT_PATH,
#     depot_locations=DEPOT_LOCATIONS,
#     receiver_distributions=RECEIVER_DISTRIBUTIONS,
#     penalty_list=PENALTY_LIST,
#     total_instances=TOTAL_INSTANCES,
#     original_tw=ORIGINAL_TW,
#     compute_network_distances=True
# )

# Option 2: Use process_all_scenarios with custom parameters (for testing with fewer scenarios)
# Test with just a subset first
test_all_metrics_df = agg_anls.process_all_scenarios(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    depot_locations=['left'],  # Test with one depot location
    receiver_distributions=['CLUSTERED'],  # Test with one distribution
    penalties=[0.0],  # Test with one penalty
    instances=[0],  # Test with first 2 instances only
    original_tw=ORIGINAL_TW,
    last_iter=LAST_ITER,
    compute_network_distances=True,
    verbose=True
)

Found 1 matching scenarios
Loading network from: /Volumes/External/gitProj/matsim-libs-2024/output/chessboardCarrierReceiverCollab/left-CLUSTERED-penSweep-af0.80-p0.0000-exactShapley-i00/output_network.xml.gz
Building network graph...
Graph built with 102 nodes and 180 edges


Processing scenarios:   0%|          | 0/1 [00:00<?, ?it/s, left-CLUSTERED-penSweep-af0.80-p0.0000-e...]/opt/anaconda3/envs/GWML/lib/python3.10/site-packages/pyogrio/geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
Processing scenarios: 100%|██████████| 1/1 [00:00<00:00,  5.23it/s, left-CLUSTERED-penSweep-af0.80-p0.0000-e...]


Processing complete. Combined metrics saved to: /Volumes/External/gitProj/matsim-libs-2024/data/freightChessboardRC/clean/all_scenarios_metrics.csv.gz


In [61]:
# View the test results
test_all_metrics_df

,depot_location,receiver_distribution,allocation_factor,penalty,instance,fleet_size,final_carrier_score,iter0_carrier_score,num_collaborative_receivers,num_non_collab_receivers,...,VKT_km,iter0_VKT_km,VTT_seconds,iter0_VTT_seconds,TKT_tonkm,iter0_TKT_tonkm,mean_receiver_dist_to_depot_euclidean_km,mean_receiver_dist_to_depot_network_km,clustering_index_euclidean_km,clustering_index_network_km
0,left,CLUSTERED,0.8,0.0,0,2,637.361974,535.983,7,3,...,49.0,58.0,20031.0,23898.0,170000.0,103000.0,6.616402,9.4,0.941421,1.2


# Read agg_file

In [14]:
agg_metrics_df = pd.read_csv(os.path.join(OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
agg_metrics_df

,depot_location,receiver_distribution,allocation_factor,penalty,instance,fleet_size,iter0_fleet_size,final_carrier_score,iter0_carrier_score,num_collaborative_receivers,...,VKT_km,iter0_VKT_km,VTT_seconds,iter0_VTT_seconds,TKT_tonkm,iter0_TKT_tonkm,mean_receiver_dist_to_depot_euclidean_km,mean_receiver_dist_to_depot_network_km,clustering_index_euclidean_km,clustering_index_network_km
0,center,CLUSTERED,0.8,0.0000,0,2,2,748.175513,587.3160,8,...,25.0,34.0,18021.0,22156.0,124000.0,75000.0,1.744697,3.0,0.941421,1.2
1,center,CLUSTERED,0.8,0.0000,1,2,2,749.426331,578.9720,9,...,25.0,34.0,18155.0,22516.0,126000.0,71000.0,2.351147,4.6,0.841421,1.0
2,center,CLUSTERED,0.8,0.0000,2,2,2,775.174101,574.7095,10,...,33.0,34.0,19365.0,22655.0,94500.0,74500.0,1.629127,2.5,0.741421,1.4
3,center,CLUSTERED,0.8,0.0000,3,2,2,682.008367,573.1600,8,...,30.0,38.0,21621.0,22516.0,61500.0,90000.0,1.932033,2.8,1.000000,1.4
4,center,CLUSTERED,0.8,0.0000,4,2,2,737.752333,573.9025,7,...,37.0,38.0,19499.0,23094.0,108500.0,80000.0,1.849190,2.8,0.941421,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754,left,FULLY_RANDOM,0.8,0.0167,26,2,2,512.316000,512.3160,1,...,62.0,62.0,25238.0,25238.0,98500.0,98500.0,5.678719,8.1,1.147871,1.9
1755,left,FULLY_RANDOM,0.8,0.0167,32,2,2,565.089341,537.1260,1,...,50.0,54.0,24032.0,24166.0,88500.0,91500.0,5.969742,8.7,0.941421,1.5
1756,left,FULLY_RANDOM,0.8,0.0167,42,2,2,534.895833,514.4600,1,...,70.0,62.0,25640.0,24970.0,117000.0,102000.0,5.857707,8.4,1.289292,1.8
1757,left,FULLY_RANDOM,0.8,0.0222,24,2,2,532.311053,502.6360,1,...,66.0,66.0,25464.0,25464.0,112000.0,112000.0,6.183579,9.2,0.523607,0.7
